In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import os


In [2]:
WAREHOUSE_DIR = f"/home/lst/my-spark/User_Behavior/spark_warehouse"
DATA_PATH = f"/home/lst/my-spark/User_Behavior/UserBehavior.csv"

In [3]:
os.makedirs(WAREHOUSE_DIR, exist_ok=True)

In [4]:
spark = SparkSession.builder\
    .master("local[4]")\
    .appName("Local_UserBehavior")\
    .config("spark.driver.memory", "4g")\
    .config("spark.sql.shuffle.partitions", "4")\
    .getOrCreate()

26/05/04 17:55:21 WARN Utils: Your hostname, LAPTOP-KVJ76CML resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/04 17:55:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/04 17:55:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
df_raw1 = spark.read.csv(DATA_PATH, header=False)
df_raw1 = df_raw1.limit(2000)
df_raw1.show()

+---+-------+-------+---+----------+
|_c0|    _c1|    _c2|_c3|       _c4|
+---+-------+-------+---+----------+
|  1|2268318|2520377| pv|1511544070|
|  1|2333346|2520771| pv|1511561733|
|  1|2576651| 149192| pv|1511572885|
|  1|3830808|4181361| pv|1511593493|
|  1|4365585|2520377| pv|1511596146|
|  1|4606018|2735466| pv|1511616481|
|  1| 230380| 411153| pv|1511644942|
|  1|3827899|2920476| pv|1511713473|
|  1|3745169|2891509| pv|1511725471|
|  1|1531036|2920476| pv|1511733732|
|  1|2266567|4145813| pv|1511741471|
|  1|2951368|1080785| pv|1511750828|
|  1|3108797|2355072| pv|1511758881|
|  1|1338525| 149192| pv|1511773214|
|  1|2286574|2465336| pv|1511797167|
|  1|5002615|2520377| pv|1511839385|
|  1|2734026|4145813| pv|1511842184|
|  1|5002615|2520377| pv|1511844273|
|  1|3239041|2355072| pv|1511855664|
|  1|4615417|4145813| pv|1511870864|
+---+-------+-------+---+----------+
only showing top 20 rows



In [6]:
df_raw = spark.read.csv(DATA_PATH, header=False, schema="""
    user_id BIGINT,
    item_id BIGINT,
    category_id BIGINT,
    behavior STRING,
    timestamp BIGINT
""") 
df_raw_temp= df_raw.limit(2000)
df_raw_temp.show()


+-------+-------+-----------+--------+----------+
|user_id|item_id|category_id|behavior| timestamp|
+-------+-------+-----------+--------+----------+
|      1|2268318|    2520377|      pv|1511544070|
|      1|2333346|    2520771|      pv|1511561733|
|      1|2576651|     149192|      pv|1511572885|
|      1|3830808|    4181361|      pv|1511593493|
|      1|4365585|    2520377|      pv|1511596146|
|      1|4606018|    2735466|      pv|1511616481|
|      1| 230380|     411153|      pv|1511644942|
|      1|3827899|    2920476|      pv|1511713473|
|      1|3745169|    2891509|      pv|1511725471|
|      1|1531036|    2920476|      pv|1511733732|
|      1|2266567|    4145813|      pv|1511741471|
|      1|2951368|    1080785|      pv|1511750828|
|      1|3108797|    2355072|      pv|1511758881|
|      1|1338525|     149192|      pv|1511773214|
|      1|2286574|    2465336|      pv|1511797167|
|      1|5002615|    2520377|      pv|1511839385|
|      1|2734026|    4145813|      pv|1511842184|


In [7]:
df_raw_temp = df_raw.withColumn("event_date", F.from_unixtime("timestamp", "yyyy-MM-dd"))
df_raw_temp.show(10)

+-------+-------+-----------+--------+----------+----------+
|user_id|item_id|category_id|behavior| timestamp|event_date|
+-------+-------+-----------+--------+----------+----------+
|      1|2268318|    2520377|      pv|1511544070|2017-11-25|
|      1|2333346|    2520771|      pv|1511561733|2017-11-25|
|      1|2576651|     149192|      pv|1511572885|2017-11-25|
|      1|3830808|    4181361|      pv|1511593493|2017-11-25|
|      1|4365585|    2520377|      pv|1511596146|2017-11-25|
|      1|4606018|    2735466|      pv|1511616481|2017-11-25|
|      1| 230380|     411153|      pv|1511644942|2017-11-26|
|      1|3827899|    2920476|      pv|1511713473|2017-11-27|
|      1|3745169|    2891509|      pv|1511725471|2017-11-27|
|      1|1531036|    2920476|      pv|1511733732|2017-11-27|
+-------+-------+-----------+--------+----------+----------+
only showing top 10 rows



In [8]:
df_raw = df_raw.withColumn("event_date", F.from_unixtime("timestamp", "yyyy-MM-dd"))
df_raw.show()


+-------+-------+-----------+--------+----------+----------+
|user_id|item_id|category_id|behavior| timestamp|event_date|
+-------+-------+-----------+--------+----------+----------+
|      1|2268318|    2520377|      pv|1511544070|2017-11-25|
|      1|2333346|    2520771|      pv|1511561733|2017-11-25|
|      1|2576651|     149192|      pv|1511572885|2017-11-25|
|      1|3830808|    4181361|      pv|1511593493|2017-11-25|
|      1|4365585|    2520377|      pv|1511596146|2017-11-25|
|      1|4606018|    2735466|      pv|1511616481|2017-11-25|
|      1| 230380|     411153|      pv|1511644942|2017-11-26|
|      1|3827899|    2920476|      pv|1511713473|2017-11-27|
|      1|3745169|    2891509|      pv|1511725471|2017-11-27|
|      1|1531036|    2920476|      pv|1511733732|2017-11-27|
|      1|2266567|    4145813|      pv|1511741471|2017-11-27|
|      1|2951368|    1080785|      pv|1511750828|2017-11-27|
|      1|3108797|    2355072|      pv|1511758881|2017-11-27|
|      1|1338525|     14

在这里只有前十行进行的time的转换

In [9]:
df_raw.count()

100150807

In [10]:
spark.sql("create database if not exists ods")
spark.sql("drop table if exists ods.ods_user_behavior")

26/05/04 17:55:31 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/05/04 17:55:31 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/05/04 17:55:32 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


DataFrame[]

In [11]:
spark.sql("show databases").show()

+--------------+
|     namespace|
+--------------+
|       default|
|      eco_data|
|           ods|
|       test_db|
|wsl_local_test|
+--------------+



In [12]:
spark.sql("use ods")
df_raw = df_raw.repartition(4)

In [13]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS ods.ods_user_behavior (
        user_id BIGINT, 
        item_id BIGINT, 
        category_id BIGINT,
        behavior STRING, 
        timestamp BIGINT
    ) 
    partitioned by (event_date STRING) 
    stored as parquet
""")

26/05/04 17:55:32 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/05/04 17:55:32 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/05/04 17:55:32 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/05/04 17:55:32 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist
26/05/04 17:55:32 WARN HiveMetaStore: Location: file:/home/lst/my-spark/User_Behavior/spark_warehouse/ods.db/ods_user_behavior specified for non-external table:ods_user_behavior


DataFrame[]

In [14]:
df_raw.write.mode("overwrite")\
    .partitionBy("event_date")\
    .format("parquet")\
    .saveAsTable("ods.ods_user_behavior")

In [15]:
spark.stop()